# Course: 65026 - Robotics and Reinforcement Learning
 **Afeka College of Engineering**  
 **Program:** Intelligent Systems  
 **Course coordinator:** Dr. Masha Fridin

**Submission for:**  Assignment 1   
**by**  
  - Michael Berger, 318063864 

# Multi-Armed Bandit with Costs  
Imagine you are managing a fleet of delivery drones, each assigned to one of several routes (arms).  
Each route provides a reward based on its efficiency (e.g., time saved or customer satisfaction), but
it also incurs a cost (e.g., fuel consumption).  
Your task is to design an algorithm that selects routes
to maximize the net reward, defined as: Net Reward = Reward – Cost.  
You will simulate the multi-armed bandit problem with costs using Python and analyze the
performance of your algorithm.



# Exercise Details
## 1. Problem Setup
- There are $k = 5$ routes (arms).
- Each route $i$ has:
  - A reward distribution: $R_i = N(\mu_i, \sigma_i^2)$
  - A fixed cost: $C_i$ , representing the cost of selecting the route.
  - The net reward for selecting arm $i$ is: $Net Reward_i = R_i - C_i$
## 2. Parameters
- Mean rewards $\mu$: `[0.8, 0.6, 0.9, 0.4, 0.7]`
- Standard deviations $\sigma$: `[0.1, 0.1, 0.1, 0.1, 0.1]`
- Costs $C$ : `[0.2, 0.1, 0.3, 0.05, 0.15]`
## 3. Goal: 
- Maximize the cumulative net reward over $T = 1000$ steps by selecting routes (arms) using an appropriate multi-armed bandit algorithm.



# Task


## Step 1: Implement the Environment
Write a function to simulate the environment:


In [1]:
from random import choice as random_choice
import numpy as np
import pandas as pd
import plotly.express as px

# Define parameters for each arm
reward_means = [0.8, 0.6, 0.9, 0.4, 0.7]
reward_stds = [0.1, 0.1, 0.1, 0.1, 0.1]
costs = [0.2, 0.1, 0.3, 0.05, 0.15]

def action(arm):
    """
    Simulates the environment by returning the reward and cost for a given arm.
    Parameters:
    arm (int): The selected arm (route).
    Returns:
    reward (float): Sampled reward from the arm's reward distribution.
    cost (float): Fixed cost of the arm.
    """
    # Sample reward from the normal distribution
    reward = np.random.normal(reward_means[arm], reward_stds[arm])
    # Return reward and cost
    return reward, costs[arm]

## Step 2: Implement the ε-Greedy Algorithm. 
Use the net reward $(R_i - C_i)$ to update the Q-values.



In [2]:

def greedy_choice(Q):
  """
  Selects the arm with the highest average reward.
  Returns:
  int: The index of the selected arm.
  """
  idx = np.argmax([np.mean(q) for q in Q.values()])
  return idx

def update_estimates(Q, a, R, t):
  Q[a] =  Q[a] + (R - Q[a]) / (t+1)
  return Q

def epsilon_greedy_with_costs(A, T, epsilon=0.05):
  RewardHistory = list(range(len(T)))
  Qhistory = list(range(len(T)))
  Q = {idx: 0 for idx in A}
  for t in T:
    Qhistory[t] = Q.copy()
    p = np.random.random()
    # explore
    if p < epsilon:
      a = random_choice(A)
    # exploit
    else:
      a = greedy_choice(Q)
    R, C = action(a)
    NetReward = R - C 
    RewardHistory[t] = NetReward
    Q = update_estimates(Q, a, NetReward, t)
    
  return RewardHistory, Qhistory

## Step 3: Run the Simulation
Run the $\epsilon$-greedy algorithm and visualize the results:


In [3]:
# Parameters
num_arms = 5
time_steps = 1000
epsilon = 0.4

A = list(range(num_arms))
T = list(range(time_steps))

# Run the algorithm
net_rewards, Q_t = epsilon_greedy_with_costs(A, T, epsilon)

# Calculate the cumulative net rewards
cumulative_net_rewards = np.cumsum(net_rewards)
times = np.array(T)
labels = np.repeat("ε-Greedy", len(cumulative_net_rewards))

df = pd.DataFrame([times, labels, net_rewards, cumulative_net_rewards ], 
                  index=["Steps", "Label","Net Reward", "Cumulative Net Reward"]).T

In [4]:


# Plot the results
plt = px.line(df, x="Steps", y="Cumulative Net Reward", color="Label",
              title="ε-Greedy Algorithm with Costs")
plt.show()

In [5]:
Q_t = pd.DataFrame(Q_t)
Q_melt = Q_t.reset_index(names=["Steps"])\
   .melt(id_vars=["Steps"], var_name="Arm", value_name="Q Value")
Q_melt
px.line(Q_melt, x = "Steps", y = "Q Value", color= "Arm" , width=800, height=400)


# Answer the questions:



## 1. Algorithm Design:
- Explain why we use net rewards $(R_i - C_i )$ instead of raw rewards $(R_i)$ in this problem.
- How does the $\epsilon$-greedy algorithm balance exploration and exploitation? What happens if epsilon is too high or too low?


## 2. Experimentation:

- Run the simulation with different values of epsilon (e.g., 0.01, 0.1, 0.5). How does the choice of epsilon affect the cumulative net reward?

In [6]:

# Parameters
num_arms = 5
time_steps = 1000
epsilons = [0.01, 0.1, 0.5, 0.7]
label = "ε-Greedy"

A = list(range(num_arms))
T = list(range(time_steps))

df = pd.DataFrame([[], [], [], []], index=["Net Reward", "Cumulative Net Reward", "Steps", "Label"]).T

tot_reward = [r-c for r,c in zip(reward_means,costs)]
gt = pd.DataFrame([tot_reward]* time_steps )
gt["Label"] = "Ground Truth"

Q_t_eps = gt

# Run the algorithm
for epsilon in epsilons:

    net_rewards, Q_t = epsilon_greedy_with_costs(A, T, epsilon)

    # Calculate the cumulative net rewards
    cumulative_net_rewards = np.cumsum(net_rewards)
    times = np.array(T)
    labels = np.repeat("ε-Greedy, ε =" + str(epsilon), len(cumulative_net_rewards))

    df = pd.concat([df, pd.DataFrame([net_rewards, cumulative_net_rewards, times, labels], 
                    index=["Net Reward", "Cumulative Net Reward", "Steps", "Label"]).T])
    Q_t_df = pd.DataFrame(Q_t)
    Q_t_df["Label"] = labels
    Q_t_eps = pd.concat([Q_t_eps, Q_t_df])

In [7]:
px.line(df, x="Steps", y="Cumulative Net Reward", color="Label",
              title="ε-Greedy Algorithm with Costs")

In [8]:
def plot_Q_evolution(Q_t):
    """
    Plots the Q-values for each arm over time.
    Parameters:
    Q_t (DataFrame): DataFrame containing Q-values for each arm.    
    """
    Q_melt = Q_t.reset_index(names=["Steps"])\
       .melt(id_vars=["Steps","Label"], var_name="Arm", value_name="Q Value")
    fig = px.line(Q_melt, x="Steps", y="Q Value", color="Arm",
                  facet_col="Label",
                  #width=800, height=400
                  title=f"Evolution of Q estimates for each arm, depending on ε")
    fig.show()

plot_Q_evolution(Q_t_eps)


- Modify the costs $(C_i)$ to make one arm significantly more expensive. How does this impact the algorithm's performance?

In [9]:
costs = [0.2, 0.1, 0.8, 0.05, 0.15]


# Parameters
num_arms = 5
time_steps = 1000
epsilons = [0.01, 0.1, 0.5, 0.7]
label = "ε-Greedy"

A = list(range(num_arms))
T = list(range(time_steps))

df = pd.DataFrame([[], [], [], []], index=["Net Reward", "Cumulative Net Reward", "Steps", "Label"]).T

tot_reward = [r-c for r,c in zip(reward_means,costs)]
gt = pd.DataFrame([tot_reward]* time_steps )
gt["Label"] = "Ground Truth"

Q_t_eps = gt

# Run the algorithm
for epsilon in epsilons:

    net_rewards, Q_t = epsilon_greedy_with_costs(A, T, epsilon)

    # Calculate the cumulative net rewards
    cumulative_net_rewards = np.cumsum(net_rewards)
    times = np.array(T)
    labels = np.repeat("ε-Greedy, ε =" + str(epsilon), len(cumulative_net_rewards))

    df = pd.concat([df, pd.DataFrame([net_rewards, cumulative_net_rewards, times, labels], 
                    index=["Net Reward", "Cumulative Net Reward", "Steps", "Label"]).T])
    Q_t_df = pd.DataFrame(Q_t)
    Q_t_df["Label"] = labels
    Q_t_eps = pd.concat([Q_t_eps, Q_t_df])

In [10]:
px.line(df, x="Steps", y="Cumulative Net Reward", color="Label",
              title="ε-Greedy Algorithm with Costs")

In [11]:
plot_Q_evolution(Q_t_eps)

## 3. Comparison:


- Non-Stationary Costs: Modify the environment so that the costs ($C_i$) change over time. For example, let $ C_i $ drift randomly every 100 steps. Update the algorithm to handle this non-stationarity.


In [14]:
costs

[0.2, 0.1, 0.8, 0.05, 0.15]

In [ ]:

def drift(costs, t, drift_period=100, scale=0.1):
    """
    adds drift to the costs of the arms.
    Args:
        t (_type_):The current time step 
        drift_period (int, optional): The period after which the costs means drift. Defaults to 100.
    """    
    if t % drift_period == 0:
        # Simulate a drift in the reward means
        costs += np.random.normal(0, scale, size=len(costs))
    return costs



    # tot_reward = [r-c for r,c in zip(reward_means,costs)]
    # return tot_reward

In [54]:

def epsilon_greedy_choice(Q):
  p = np.random.random()
  # explore
  if p < epsilon:
    a = random_choice(list(Q.keys()))
  # exploit
  else:
    a = greedy_choice(Q)
  return a

def make_choice(Q):
  return epsilon_greedy_choice(Q)

def simulate(A, T, epsilon=0.05):

  global costs
  GThistory = list(range(len(T)))
  RewardHistory = list(range(len(T)))
  Qhistory = list(range(len(T)))
  Q = {idx: 0 for idx in A}
  for t in T:
    Qhistory[t] = Q.copy()
    costs = drift(costs, t)
    GThistory[t] = [r-c for r,c in zip(reward_means,costs)]
    a = make_choice(Q)
    R, C = action(a)
    NetReward = R - C 
    RewardHistory[t] = NetReward
    Q = update_estimates(Q, a, NetReward, t)
    
  return RewardHistory, Qhistory, GThistory

In [55]:
wat = Q_t[0].keys()
list(wat)
# Parameters

[0, 1, 2, 3, 4]

In [56]:
costs = [0.2, 0.1, 0.3, 0.05, 0.15]


# Parameters
num_arms = 5
time_steps = 1000
epsilons = [0.01, 0.1, 0.5, 0.7]
label = "ε-Greedy"

A = list(range(num_arms))
T = list(range(time_steps))

df = pd.DataFrame([[], [], [], []], index=["Net Reward", "Cumulative Net Reward", "Steps", "Label"]).T

#tot_reward = [r-c for r,c in zip(reward_means,costs)]


Q_t_eps = pd.DataFrame([])

# Run the algorithm
for epsilon in epsilons:

    net_rewards, Q_t, GT_t = simulate(A, T, epsilon)

    # Calculate the cumulative net rewards
    cumulative_net_rewards = np.cumsum(net_rewards)
    times = np.array(T)
    labels = np.repeat("ε-Greedy, ε =" + str(epsilon), len(cumulative_net_rewards))

    df = pd.concat([df, pd.DataFrame([net_rewards, cumulative_net_rewards, times, labels], 
                    index=["Net Reward", "Cumulative Net Reward", "Steps", "Label"]).T])
    Q_t_df = pd.DataFrame(Q_t)

    gt = pd.DataFrame(GT_t)
    gt["Label"] = "Ground Truth"

    Q_t_df["Label"] = labels
    Q_t_eps = pd.concat([Q_t_eps, Q_t_df, gt])

In [57]:
Q_t_eps

,0,1,2,3,4,Label
0,0.000000,0.000000,0.000000,0.000000,0.000000,"ε-Greedy, ε =0.01"
1,0.484441,0.000000,0.000000,0.000000,0.000000,"ε-Greedy, ε =0.01"
2,0.467346,0.000000,0.000000,0.000000,0.000000,"ε-Greedy, ε =0.01"
3,0.418191,0.000000,0.000000,0.000000,0.000000,"ε-Greedy, ε =0.01"
4,0.473045,0.000000,0.000000,0.000000,0.000000,"ε-Greedy, ε =0.01"
...,...,...,...,...,...,...
995,1.016398,1.334126,0.167541,-0.460207,0.600067,Ground Truth
996,1.016398,1.334126,0.167541,-0.460207,0.600067,Ground Truth
997,1.016398,1.334126,0.167541,-0.460207,0.600067,Ground Truth
998,1.016398,1.334126,0.167541,-0.460207,0.600067,Ground Truth


In [58]:
plot_Q_evolution(Q_t_eps)

In [ ]:
px.line(df, x="Steps", y="Cumulative Net Reward", color="Label",
              title="ε-Greedy Algorithm with Costs")

- Implement UCB algorithm and compare its performance with $\epsilon$-greedy. Which algorithm performs better in terms of cumulative net reward?
- Dynamic Rewards: Introduce changes in the reward distributions $(R_i)$ over time. For example, let the mean rewards ($\mu_i$) shift periodically. Analyze how the algorithm adapts to these changes.


## 4. Analysis:
- Calculate the regret for the $\epsilon$-greedy algorithm. Regret is defined as the difference between the cumulative net reward of the optimal arm and the cumulative net reward obtained by the algorithm.
- Discuss how the regret grows over time. Is the growth linear or sublinear?